# Modelo de Poisson para predicción de partidos — Fase de Grupos

Este notebook aplica una distribución de Poisson para estimar la probabilidad de diferentes marcadores en cada partido, calculando:
- Probabilidad de victoria local, empate y victoria visitante
- Ganador predicho (o empate)
- Los dos marcadores más probables con sus respectivas probabilidades

## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
import math
from itertools import product
import warnings
warnings.filterwarnings('ignore')

## 2. Carga de datos

In [3]:
df = pd.read_csv('Fase grupos.txt')

# Extraer nombres de equipos desde la columna 'Partido'
df['Equipo_A'] = df['Partido'].apply(lambda x: x.split('_vs_')[0])
df['Equipo_B'] = df['Partido'].apply(lambda x: x.split('_vs_')[1])

print(f'Partidos cargados: {len(df)}')
df.head()

Partidos cargados: 72


,Partido,Sede,OV_A,DV_A,MV_A,FA_A,OV_B,DV_B,MV_B,FA_B,Equipo_A,Equipo_B
0,México_vs_Sudáfrica,Ciudad de México (Altitud),84,73,227.99,1.00,53,46,23.4,0.80,México,Sudáfrica
1,Corea del Sur_vs_República Checa,Guadalajara (Altitud),76,59,185.50,0.85,68,66,165.3,0.85,Corea del Sur,República Checa
2,Canadá_vs_Bosnia y Herzegovina,Toronto (Neutro),74,68,151.00,1.00,70,58,46.5,0.95,Canadá,Bosnia y Herzegovina
3,Estados Unidos_vs_Paraguay,Los Angeles (Neutro),80,74,387.30,1.00,73,72,115.4,0.95,Estados Unidos,Paraguay
4,Qatar_vs_Suiza,Santa Clara (Neutro),59,51,19.80,0.90,81,88,359.3,0.95,Qatar,Suiza


## 3. Construcción de la Lambda

La lambda de la Poisson se construye combinando las tres variables de cada selección:

$$\lambda_A = \left(\frac{OV_A}{100}\right)^{\alpha} \times \left(\frac{100}{DV_B}\right)^{\beta} \times \left(\frac{MV_A}{MV_{ref}}\right)^{\gamma} \times FA_A \times k$$

Donde:
- **OV_A / 100**: capacidad ofensiva normalizada del equipo A
- **100 / DV_B**: vulnerabilidad defensiva del rival (mayor DV rival → menos goles esperados)
- **MV_A / MV_ref**: peso relativo del valor de mercado respecto al promedio del torneo
- **FA_A**: factor de aclimatación del equipo A en esa sede
- **k**: constante de escala para que el promedio de goles sea realista (~1.3 goles/equipo/partido)
- **α=0.6, β=0.3, γ=0.1**: ponderaciones (más peso al ataque, algo de defensa rival, poco al valor mercado como corrector de calidad)

In [4]:
# Parámetros del modelo
alpha = 0.6   # peso del Valor Ofensivo
beta  = 0.3   # peso del Valor Defensivo rival
gamma = 0.1   # peso del Valor de Mercado
k     = 1.5   # constante de escala de goles esperados

# MV de referencia: promedio de todos los valores de mercado del torneo
MV_ref = pd.concat([df['MV_A'], df['MV_B']]).mean()
print(f'Valor de Mercado de referencia (MV_ref): {MV_ref:.2f} M€')

# Cálculo de lambdas
df['lambda_A'] = (
    (df['OV_A'] / 100) ** alpha *
    (100 / df['DV_B'])  ** beta  *
    (df['MV_A'] / MV_ref) ** gamma *
    df['FA_A']
) * k

df['lambda_B'] = (
    (df['OV_B'] / 100) ** alpha *
    (100 / df['DV_A'])  ** beta  *
    (df['MV_B'] / MV_ref) ** gamma *
    df['FA_B']
) * k

print(f"\nLambda A — Media: {df['lambda_A'].mean():.3f} | Min: {df['lambda_A'].min():.3f} | Max: {df['lambda_A'].max():.3f}")
print(f"Lambda B — Media: {df['lambda_B'].mean():.3f} | Min: {df['lambda_B'].min():.3f} | Max: {df['lambda_B'].max():.3f}")

df[['Partido', 'Equipo_A', 'Equipo_B', 'lambda_A', 'lambda_B']].head(10)

Valor de Mercado de referencia (MV_ref): 367.72 M€

Lambda A — Media: 1.329 | Min: 0.654 | Max: 2.109
Lambda B — Media: 1.137 | Min: 0.608 | Max: 1.856


,Partido,Equipo_A,Equipo_B,lambda_A,lambda_B
0,México_vs_Sudáfrica,México,Sudáfrica,1.625816,0.684100
1,Corea del Sur_vs_República Checa,Corea del Sur,República Checa,1.143979,1.094046
2,Canadá_vs_Bosnia y Herzegovina,Canadá,Bosnia y Herzegovina,1.348801,1.050303
3,Estados Unidos_vs_Paraguay,Estados Unidos,Paraguay,1.455454,1.150027
4,Qatar_vs_Suiza,Qatar,Suiza,0.763159,1.533306
5,Brasil_vs_Marruecos,Brasil,Marruecos,1.527743,1.265677
6,Haití_vs_Escocia,Haití,Escocia,0.676983,1.419919
7,Australia_vs_Turquía,Australia,Turquía,0.971934,1.568488
8,Alemania_vs_Curazao,Alemania,Curazao,1.822941,0.619015
9,Países Bajos_vs_Japón,Países Bajos,Japón,1.579113,1.186427


## 4. Función de Poisson

$$P(X = x \mid \lambda) = \frac{e^{-\lambda} \cdot \lambda^x}{x!}$$

In [5]:
def poisson_prob(lambd: float, n_goals: int) -> float:
    """Calcula la probabilidad de anotar exactamente n_goals goles dado un lambda.
    
    >>> poisson_prob(1.5, 2)
    0.2510...
    """
    return (math.exp(-lambd) * (lambd ** n_goals)) / math.factorial(n_goals)


# Rango de goles: 0 a 7
GOL_MIN = 0
GOL_MAX = 7

# Verificación rápida
ejemplo_lam = 1.5
print(f'Probabilidades para λ={ejemplo_lam}:')
for g in range(GOL_MIN, GOL_MAX + 1):
    print(f'  P(X={g}) = {poisson_prob(ejemplo_lam, g):.4f}')

Probabilidades para λ=1.5:
  P(X=0) = 0.2231
  P(X=1) = 0.3347
  P(X=2) = 0.2510
  P(X=3) = 0.1255
  P(X=4) = 0.0471
  P(X=5) = 0.0141
  P(X=6) = 0.0035
  P(X=7) = 0.0008


## 5. Cálculo de probabilidades por partido

Para cada partido se calculan:
- La matriz de probabilidades de todos los posibles marcadores (0-0 hasta 7-7)
- P(Victoria A): suma de celdas donde goles_A > goles_B
- P(Empate): suma de la diagonal
- P(Victoria B): suma de celdas donde goles_B > goles_A
- Los dos marcadores más probables

In [6]:
def analizar_partido(lambda_a: float, lambda_b: float,
                     equipo_a: str, equipo_b: str,
                     gol_min: int = 0, gol_max: int = 7) -> dict:
    """
    Dado lambda_a y lambda_b, calcula:
    - P_A, P_E, P_B (probabilidades de victoria A, empate, victoria B)
    - Ganador predicho
    - Top-2 marcadores más probables con sus probabilidades
    """
    goles = range(gol_min, gol_max + 1)
    
    # Probabilidades marginales
    prob_a = np.array([poisson_prob(lambda_a, g) for g in goles])
    prob_b = np.array([poisson_prob(lambda_b, g) for g in goles])
    
    # Matriz de marcadores: fila = goles_A, columna = goles_B
    matriz = np.outer(prob_a, prob_b)
    
    # Probabilidades de resultado
    P_A = float(np.sum(np.tril(matriz, k=-1)))  # goles_A > goles_B
    P_E = float(np.sum(np.diag(matriz)))         # goles_A == goles_B
    P_B = float(np.sum(np.triu(matriz, k=1)))    # goles_B > goles_A
    
    # Ganador predicho
    max_p = max(P_A, P_E, P_B)
    if max_p == P_A:
        ganador = equipo_a
    elif max_p == P_B:
        ganador = equipo_b
    else:
        ganador = 'Empate'
    
    # Top-2 marcadores más probables
    scores = []
    for i, ga in enumerate(goles):
        for j, gb in enumerate(goles):
            scores.append((f'{ga}-{gb}', round(float(matriz[i, j]), 6)))
    scores.sort(key=lambda x: x[1], reverse=True)
    
    return {
        'P_Victoria_A': round(P_A, 4),
        'P_Empate':     round(P_E, 4),
        'P_Victoria_B': round(P_B, 4),
        'Ganador_Pred': ganador,
        'Marcador_1':      scores[0][0],
        'Prob_Marcador_1': scores[0][1],
        'Marcador_2':      scores[1][0],
        'Prob_Marcador_2': scores[1][1],
    }

print('Función definida correctamente.')

Función definida correctamente.


## 6. Aplicar el modelo a todos los partidos

In [7]:
resultados = []

for _, row in df.iterrows():
    res = analizar_partido(
        lambda_a  = row['lambda_A'],
        lambda_b  = row['lambda_B'],
        equipo_a  = row['Equipo_A'],
        equipo_b  = row['Equipo_B'],
        gol_min   = GOL_MIN,
        gol_max   = GOL_MAX
    )
    resultados.append(res)

resultados_df = pd.DataFrame(resultados)

# Unir con el dataframe original
final = pd.concat([
    df[['Partido', 'Sede', 'Equipo_A', 'Equipo_B',
        'OV_A', 'DV_A', 'MV_A', 'FA_A',
        'OV_B', 'DV_B', 'MV_B', 'FA_B',
        'lambda_A', 'lambda_B']].reset_index(drop=True),
    resultados_df
], axis=1)

print(f'Partidos procesados: {len(final)}')
final[['Partido', 'lambda_A', 'lambda_B',
       'P_Victoria_A', 'P_Empate', 'P_Victoria_B',
       'Ganador_Pred', 'Marcador_1', 'Prob_Marcador_1',
       'Marcador_2', 'Prob_Marcador_2']].head(10)

Partidos procesados: 72


,Partido,lambda_A,lambda_B,P_Victoria_A,P_Empate,P_Victoria_B,Ganador_Pred,Marcador_1,Prob_Marcador_1,Marcador_2,Prob_Marcador_2
0,México_vs_Sudáfrica,1.625816,0.684100,0.6005,0.2444,0.1547,México,1-0,0.161394,2-0,0.131198
1,Corea del Sur_vs_República Checa,1.143979,1.094046,0.3683,0.2882,0.3434,Corea del Sur,1-1,0.133503,1-0,0.122027
2,Canadá_vs_Bosnia y Herzegovina,1.348801,1.050303,0.4356,0.2728,0.2914,Canadá,1-1,0.128631,1-0,0.122470
3,Estados Unidos_vs_Paraguay,1.455454,1.150027,0.4411,0.2599,0.2988,Estados Unidos,1-1,0.123640,1-0,0.107511
4,Qatar_vs_Suiza,0.763159,1.533306,0.1859,0.2576,0.5563,Suiza,0-1,0.154272,0-2,0.118273
5,Brasil_vs_Marruecos,1.527743,1.265677,0.4338,0.2507,0.3153,Brasil,1-1,0.118360,1-0,0.093515
6,Haití_vs_Escocia,0.676983,1.419919,0.1779,0.2725,0.5495,Escocia,0-1,0.174418,0-2,0.123830
7,Australia_vs_Turquía,0.971934,1.568488,0.2343,0.2534,0.5121,Turquía,0-1,0.123649,1-1,0.120178
8,Alemania_vs_Curazao,1.822941,0.619015,0.6631,0.2166,0.1197,Alemania,1-0,0.158579,2-0,0.144540
9,Países Bajos_vs_Japón,1.579113,1.186427,0.4642,0.2490,0.2865,Países Bajos,1-1,0.117922,1-0,0.099393


## 7. Revisión de predicciones

In [8]:
print('=== Distribución de resultados predichos ===')
print(final['Ganador_Pred'].value_counts().head(20))

print('\n=== Marcadores más frecuentes (Marcador_1) ===')
print(final['Marcador_1'].value_counts())

print('\n=== Probabilidades promedio ===')
print(f"P_Victoria_A promedio : {final['P_Victoria_A'].mean():.3f}")
print(f"P_Empate     promedio : {final['P_Empate'].mean():.3f}")
print(f"P_Victoria_B promedio : {final['P_Victoria_B'].mean():.3f}")

=== Distribución de resultados predichos ===
Ganador_Pred
México            3
Alemania          3
Portugal          3
Inglaterra        3
Francia           3
Bélgica           3
España            3
Países Bajos      3
Argentina         3
Turquía           3
Brasil            3
Suiza             3
Austria           2
Croacia           2
Japón             2
Marruecos         2
Colombia          2
Ecuador           2
Estados Unidos    2
Canadá            2
Name: count, dtype: int64

=== Marcadores más frecuentes (Marcador_1) ===
Marcador_1
1-0    25
1-1    24
0-1    19
2-0     2
0-0     2
Name: count, dtype: int64

=== Probabilidades promedio ===
P_Victoria_A promedio : 0.414
P_Empate     promedio : 0.256
P_Victoria_B promedio : 0.329


## 8. Exportar a CSV

In [ ]:
output_cols = [
    'Partido', 'Sede', 'Equipo_A', 'Equipo_B',
    'OV_A', 'DV_A', 'MV_A', 'FA_A',
    'OV_B', 'DV_B', 'MV_B', 'FA_B',
    'lambda_A', 'lambda_B',
    'P_Victoria_A', 'P_Empate', 'P_Victoria_B',
    'Ganador_Pred',
    'Marcador_1', 'Prob_Marcador_1',
    'Marcador_2', 'Prob_Marcador_2'
]

output_path = 'predicciones_fase_grupos.csv'
final[output_cols].to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'CSV exportado: {output_path}')
print(f'Filas: {len(final)} | Columnas: {len(output_cols)}')
final[output_cols].head()

---
**Elaborado con modelo de Poisson**  
Variables: Valor Ofensivo (OV), Valor Defensivo (DV), Valor de Mercado (MV), Factor de Aclimatación (FA)  
λ = (OV/100)^0.6 × (100/DV_rival)^0.3 × (MV/MV_ref)^0.1 × FA × 1.5  
Rango Poisson: 0 – 7 goles